## Quora Question Pairs: Train Models

This notebook:
1. Trains models ```baseline``` and ```enhanced```
2. Saves them to ```models/```.  

It is reproducible, meaning that running it twice produces the same results.  
No functions are defined here, all logic is defined in ```utils.py```.

In [25]:
from pathlib import Path
import pickle
import pandas as pd
import sklearn
import sklearn.model_selection
import sklearn.feature_extraction.text
import sklearn.linear_model
import sklearn.ensemble
import importlib
import utils
importlib.reload(utils)


<module 'utils' from '/home/ntorquet/Documents/llm/TorquetLunaNuria/utils.py'>

## 1. Load data and create splits
Data is read from ```$HOME/Datasets/QuoraQuestionPairs/quora_data.csv```.  
Splits use fixed ```random_state=123``` for reproducibility.

In [9]:
# Split data into train, validation and test sets used by both approaches
data_path = Path("~/Datasets/QuoraQuestionPairs/quora_data.csv").expanduser()
quora_df = pd.read_csv(data_path)

A_df, test_df = sklearn.model_selection.train_test_split(
    quora_df, test_size=0.05, random_state=123
)
train_df, val_df = sklearn.model_selection.train_test_split(
    A_df, test_size=0.05, random_state=123
)

print("train_df.shape=", train_df.shape)
print("val_df.shape=", val_df.shape)
print("test_df.shape=", test_df.shape)


train_df.shape= (291897, 6)
val_df.shape= (15363, 6)
test_df.shape= (16172, 6)


In [10]:
train_df[20:30]


,id,qid1,qid2,question1,question2,is_duplicate
221406,3341,6623,6624,What do you hate about Toptal's interviewing p...,I need an Indian representative for the supply...,0
225507,132567,82504,212232,What are dimension work?,What dimensions are these?,0
264658,365357,399282,337374,Which is better MacBook or MacBook Pro or MacB...,Is a MacBook Pro worth the money when I can bu...,0
159282,48685,86767,86768,"I wrote a letter, asking for donations from co...","I wrote a letter, asking companies and philant...",1
67672,46012,57994,8120,How do I learn how to invest in stock market a...,What is the best way to learn about stock mark...,1
35861,149668,235708,1672,Who do you love and why?,What do you love? Why?,0
320204,87291,147041,147042,What is currently happening in Turkey?,What's happening in Turkey?,1
259492,183173,110781,38756,Can someone still get your message on snapchat...,If I block someone on snapchat will they still...,0
160754,334925,462157,462158,How long does it take for a human body to comp...,How would it take for a human body to decay to...,0
269562,238681,145570,291752,What is the most inspiring movie you have ever...,What are your most inspiring movies?,1


In [11]:
# store the splits in csv files
Path("data/splits").mkdir(parents=True, exist_ok=True)
train_df.to_csv("data/splits/train_data.csv", index=False)
val_df.to_csv("data/splits/val_data.csv", index=False)
test_df.to_csv("data/splits/test_data.csv", index=False)
print("Data splits stored in data/splits/")


Data splits stored in data/splits/


## 2. Check models folder: skip training if model exists
From the specification in the guidelines pdf file: *'The code should check if the folder is there, and in such case do not overwrite/store the models.'*  
So first we will check if the pkl files already exist in the model's folder.  
In case they exist, do not execute notebook. Otherwise, start training to generate the files.

In [ ]:
MODELS_DIR = Path("models")
SIMPLE_MODEL_PATH = MODELS_DIR / "simple_model.pkl"
ENHANCED_MODEL_PATH = MODELS_DIR / "enhanced_model.pkl"
VECTORIZER_PATH = MODELS_DIR / "count_vectorizer.pkl"
GRAPH_PATH = MODELS_DIR / "graph_features.pkl"

models_exist = all(
    path.exists() for path in [SIMPLE_MODEL_PATH, ENHANCED_MODEL_PATH, VECTORIZER_PATH, GRAPH_PATH]
)

models_exist = False # FIXME: Correct when submission is ready

if models_exist:
    print("Models and vectorizer already exist. Skipping training.")
    print("To retrain, delete the files in the models directory.")
else:
    Path("models").mkdir(parents=True, exist_ok=True)
    print("Training models...")

Training models...


## 3. Simple model (baseline): CountVectorizer + Logistic Regression

In [13]:
if not models_exist:
    # Build vocabulary from training questions only
    q1_train = utils.cast_list_as_strings(list(train_df["question1"]))
    q2_train = utils.cast_list_as_strings(list(train_df["question2"]))
    all_quesions_train = q1_train + q2_train

    count_vectorizer = sklearn.feature_extraction.text.CountVectorizer(ngram_range=(1, 1))
    count_vectorizer.fit(all_quesions_train)
    print(f"Vocabulary size: {len(count_vectorizer.vocabulary_)}")

    # Build feature matrices
    X_train = utils.get_features_from_df(train_df, count_vectorizer)
    y_train = train_df["is_duplicate"].values

    # Train
    simple_model = sklearn.linear_model.LogisticRegression(solver="liblinear", random_state=123)
    simple_model.fit(X_train, y_train)
    print("Simple model trained.")

    # Save models and vectorizer
    with open(SIMPLE_MODEL_PATH, "wb") as f:
        pickle.dump(simple_model, f)
    with open(VECTORIZER_PATH, "wb") as f:
        pickle.dump(count_vectorizer, f)
    print(f"Saved: {SIMPLE_MODEL_PATH}, {VECTORIZER_PATH}")


Vocabulary size: 74825
Simple model trained.
Saved: models/simple_model.pkl, models/count_vectorizer.pkl


## 4. Enhanced model: 16 features + Gradient Boosting

### 4.1. Features


| # | Feature | Category | Description |
|---|---------|----------|-------------|
| 1 | `jaccard` | Statistical | Word set overlap |
| 2 | `jaccard_no_stopwords` | Statistical | Content-word set overlap |
| 3 | `shared_word_ratio` | Statistical | Dice on word bags |
| 4 | `shared_word_ratio_no_stopwords` | Statistical | Dice on content-word bags |
| 5 | `char_trigram_similarity` | Statistical | Character 3-gram Jaccard |
| 6 | `length_diff_ratio` | Statistical | Normalised length gap |
| 7 | `tfidf_cosine` | NLP | TF-IDF cosine (all words) |
| 8 | `tfidf_cosine_no_stopwords` | NLP | TF-IDF cosine (content words) |
| 9 | `lcs_ratio` | NLP | LCS / avg_length (all words) |
| 10 | `lcs_ratio_no_stopwords` | NLP | LCS / avg_length (content words) |
| 11 | `q1_freq` | Graph | Node degree of q1 |
| 12 | `q2_freq` | Graph | Node degree of q2 |
| 13 | `freq_diff` | Graph | \|q1_freq - q2_freq\| |
| 14 | `freq_sum` | Graph | q1_freq + q2_freq |
| 15 | `freq_min` | Graph | min(q1_freq, q2_freq) |
| 16 | `pair_count` | Graph | Times this exact pair appears |

In [23]:
if not models_exist:
    print("Computing graph features")
    q_freq, pair_freq = utils.build_graph_features(train_df)
    print(f"Unique questions seen during training: {len(q_freq)}")
    print(f"Unique question pairs seen during training: {len(pair_freq)}")

    # Most frequent questions are treated as hub nodes
    top10 = sorted(q_freq.items(), key=lambda x: -x[1])[:10]
    print("Top 10 most frequent questions:")
    for q, freq in top10:
        print(f"  {q} (freq={freq})")
    
    with open(GRAPH_PATH, "wb") as f:
        pickle.dump({"q_freq": q_freq, "pair_freq": pair_freq}, f)
    print(f"Saved graph features to {GRAPH_PATH}")

Computing graph features
Unique questions seen during training: 413822
Unique question pairs seen during training: 583794
Top 10 most frequent questions:
  What are the best ways to lose weight? (freq=129)
  How can I lose weight quickly? (freq=83)
  How can you look at someone's private Instagram account without following them? (freq=81)
  What's the easiest way to make money online? (freq=64)
  Can you see who views your Instagram? (freq=61)
  What are some things new employees should know going into their first day at AT&T? (freq=52)
  How can you increase your height? (freq=51)
  What do you think of the decision by the Indian Government to demonetize 500 and 1000 rupee notes? (freq=47)
  Which is the best digital marketing course? (freq=47)
  How do l see who viewed my videos on Instagram? (freq=47)
Saved graph features to models/graph_features.pkl


In [26]:
if not models_exist:
    print("Computing statistical and NLP features")
    X_train_feat = utils.get_features(train_df, q_freq, pair_freq)
    y_train = train_df["is_duplicate"].values
    print("Statistical and NLP features computed.")
    print(f"Features matrix shape: {X_train_feat.shape}")

Computing statistical and NLP features
Statistical and NLP features computed.
Features matrix shape: (291897, 16)


In [27]:
if not models_exist:
    enhanced_model = sklearn.ensemble.GradientBoostingClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        random_state=123
    )
    enhanced_model.fit(X_train_feat, y_train)
    print("Enhanced model trained.")

    print("Feature importances:")
    for name, imp in sorted(zip(utils.FEATURE_NAMES, enhanced_model.feature_importances_), key=lambda x: -x[1]):
        bar = "*" * int(imp * 200)
        print(f"{name:38s} {imp:.4f} {bar}")
    
    with open(ENHANCED_MODEL_PATH, "wb") as f:
        pickle.dump(enhanced_model, f)
    print(f"Saved: {ENHANCED_MODEL_PATH}")

Enhanced model trained.
Feature importances:
freq_min                               0.5404 ************************************************************************************************************
char_trigram_similarity                0.1370 ***************************
jaccard_no_stopwords                   0.0999 *******************
shared_word_ratio_no_stopwords         0.0541 **********
lcs_ratio_no_stopwords                 0.0389 *******
lcs_ratio                              0.0371 *******
length_diff_ratio                      0.0227 ****
tfidf_cosine_no_stopwords              0.0165 ***
freq_diff                              0.0136 **
freq_sum                               0.0123 **
tfidf_cosine                           0.0078 *
jaccard                                0.0070 *
q2_freq                                0.0059 *
shared_word_ratio                      0.0038 
q1_freq                                0.0027 
pair_count                             0.0000 
Saved: mode